# Mess food demand: model training

Trains several regression models to predict **`amount_of_food_required_kg`** from
`day_of_week`, `meal_type`, `menu` and `is_weekend`.

* Reads `ml/data/mess_food_dataset.csv`
* Saves every trained model in `ml/models/`
* **Last section:** copies the model you choose to `backend/model/model.joblib`
  (the fixed name the API loads), plus `menus_by_meal.json` and `model_info.json`

Run the cells from top to bottom. The notebook finds the project folder by itself, so it
works no matter where Jupyter / VS Code starts it from.

In [1]:
import json
import shutil
import sys
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

## 1. Paths

In [3]:
def find_project_root():
    # Walk up from the current folder until we find the one that has ml/ and backend/
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "ml").is_dir() and (folder / "backend").is_dir():
            return folder
    raise FileNotFoundError(
        "Open this notebook from inside the project folder (the one that contains ml/ and backend/)."
    )


ROOT = find_project_root()

DATA_PATH = ROOT / "ml" / "data" / "mess_food_dataset.csv"
MODELS_DIR = ROOT / "ml" / "models"      # every trained model is saved here
DEPLOY_DIR = ROOT / "backend" / "model"  # the one model the API loads

MODELS_DIR.mkdir(parents=True, exist_ok=True)
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)

Project root: C:\Users\nitin\Desktop\PROJECTS\smart-food-management


## 2. Data

In [4]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(10000, 5)


,day_of_week,meal_type,menu,is_weekend,amount_of_food_required_kg
0,Monday,Breakfast,"Aloo paratha, curd, pickle",0,213.0
1,Monday,Lunch,"Kadhi pakora, rice, roti",0,427.9
2,Monday,Dinner,"Kofta curry, roti, rice, salad",0,366.7
3,Tuesday,Breakfast,"Poha, tea, banana",0,215.4
4,Tuesday,Lunch,"Paneer curry, rice, roti, salad",0,366.7


In [5]:
# Input features and target
features = [
    "day_of_week",
    "meal_type",
    "menu",
    "is_weekend"
]

In [6]:
target = "amount_of_food_required_kg"

In [7]:
X = df[features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 3. Preprocessing and models

In [8]:
# Preprocess categorical and numerical inputs
categorical = ["day_of_week", "meal_type", "menu"]
numerical = [
    "is_weekend"
]

In [9]:
preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical),

    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numerical)
])

In [10]:
# Models
models = {
    "Linear_Regression": LinearRegression(),

    "Ridge": Ridge(alpha=10.0),

    "Lasso": Lasso(alpha=0.001, max_iter=20000),

    "KNN": KNeighborsRegressor(
        n_neighbors=7,
        weights="distance",
        p=2
    ),

    "SVR": SVR(
        kernel="rbf",
        C=100,
        gamma="scale",
        epsilon=0.1
    ),

    "Random_Forest": RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features=1.0,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient_Boosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=2,
        min_samples_split=4,
        min_samples_leaf=2,
        loss="huber",
        random_state=42
    )
}

In [11]:
results = []

## 4. Train, evaluate and save every model to `ml/models/`

In [12]:
# Train, evaluate and save each model
for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    # Accuracy-like score based on relative absolute error
    accuracy = (1 - mae / y_test.abs().mean()) * 100

    results.append({
        "Model": name,
        "MAE_kg": round(mae, 3),
        "R2": round(r2, 3),
        "Accuracy_%": round(accuracy, 2)
    })

    joblib.dump(pipeline, MODELS_DIR / f"{name}.joblib")

    print(
        f"{name}: "
        f"Accuracy={accuracy:.2f}%, "
        f"MAE={mae:.3f} kg, "
        f"R2={r2:.3f}"
    )

Linear_Regression: Accuracy=92.49%, MAE=23.605 kg, R2=0.892
Ridge: Accuracy=92.49%, MAE=23.601 kg, R2=0.892
Lasso: Accuracy=92.49%, MAE=23.605 kg, R2=0.892
KNN: Accuracy=92.08%, MAE=24.880 kg, R2=0.877
SVR: Accuracy=92.34%, MAE=24.050 kg, R2=0.888
Random_Forest: Accuracy=92.46%, MAE=23.689 kg, R2=0.891
Gradient_Boosting: Accuracy=92.55%, MAE=23.391 kg, R2=0.894


In [13]:
# Compare model performance
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("MAE_kg")

print("\nModel comparison:")
print(results_df.to_string(index=False))

results_df.to_csv(MODELS_DIR / "model_comparison.csv", index=False)
print(f"\nAll models saved in {MODELS_DIR.relative_to(ROOT)}")


Model comparison:
            Model  MAE_kg    R2  Accuracy_%
Gradient_Boosting  23.391 0.894       92.55
            Ridge  23.601 0.892       92.49
            Lasso  23.605 0.892       92.49
Linear_Regression  23.605 0.892       92.49
    Random_Forest  23.689 0.891       92.46
              SVR  24.050 0.888       92.34
              KNN  24.880 0.877       92.08

All models saved in ml\models


## 5. Deploy: copy one model to `backend/model/`

Change `DEPLOY_MODEL` to any name from the table above and run the next two cells.
The file is always saved as **`model.joblib`**, so `app.py` never needs to change when you
switch models. Commit `backend/model/` and push to redeploy.

In [14]:
DEPLOY_MODEL = "KNN"   # <- change this to deploy a different model

deployed_file = DEPLOY_DIR / "model.joblib"   # fixed name: app.py always loads this file
shutil.copy2(MODELS_DIR / f"{DEPLOY_MODEL}.joblib", deployed_file)

# Menus served at each meal (the website's dropdowns use this file)
menus_by_meal = {
    meal: sorted(df.loc[df["meal_type"] == meal, "menu"].unique().tolist())
    for meal in ["Breakfast", "Lunch", "Dinner"]
}
with open(DEPLOY_DIR / "menus_by_meal.json", "w", encoding="utf-8") as f:
    json.dump(menus_by_meal, f, indent=2, ensure_ascii=False)

# A record of exactly what is deployed (model, scores, data, library versions)
scores = results_df[results_df["Model"] == DEPLOY_MODEL].iloc[0]
model_info = {
    "model_name": DEPLOY_MODEL,
    "file": "model.joblib",
    "target": target,
    "features": features,
    "metrics_on_test_set": {
        "mae_kg": float(scores["MAE_kg"]),
        "r2": float(scores["R2"]),
        "accuracy_percent": float(scores["Accuracy_%"]),
    },
    "training": {
        "dataset": DATA_PATH.name,
        "rows": int(len(df)),
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "split": "80/20, random_state=42",
    },
    "trained_on": datetime.now().strftime("%Y-%m-%d"),
    "library_versions": {
        "python": sys.version.split()[0],
        "scikit-learn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "joblib": joblib.__version__,
    },
}
with open(DEPLOY_DIR / "model_info.json", "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=2)

print(f"Deployed {DEPLOY_MODEL} -> {deployed_file.relative_to(ROOT)}")
print(f"Also wrote menus_by_meal.json and model_info.json in {DEPLOY_DIR.relative_to(ROOT)}")

Deployed KNN -> backend\model\model.joblib
Also wrote menus_by_meal.json and model_info.json in backend\model


In [15]:
# Quick check: load the deployed file the same way the API does and predict once
deployed = joblib.load(deployed_file)
sample = pd.DataFrame([{
    "day_of_week": "Wednesday",
    "meal_type": "Breakfast",
    "menu": menus_by_meal["Breakfast"][0],
    "is_weekend": 0,
}])
print(f"Sample prediction: {deployed.predict(sample)[0]:.1f} kg")

Sample prediction: 236.3 kg
